## Data integration + feature engineering (student-semester grain)

Goal: build a single modeling table at **1 row per student per semester** with features derived from:
- Attendance trends (`student_attendance_*`)
- Tuition/fees payment trends (`student_payments_*`)
- Sponsorship trends (`student_sponsorships_*`)
- Past performance (`student_transcript_*`, `fact_student_academic_performance_*`)
- Optional enrichment: high school tier/ownership (`student_high_schools_all`), students list (`students_list15/16.xlsx`), academic progression (`academic_progression_list15/16.xlsx`)

Target variable: **`CGPA`** (from `student_transcript_*`).

**Data loading:** For each table we prefer CSV when present, then fall back to Excel (`.xlsx`), matching the profiling notebook so the pipeline works with either format.

Output: `outputs/modeling_table_student_semester.csv`

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)


def load_table(data_dir: Path, stem: str):
    """Load a single table: prefer CSV, then Excel (first sheet). Returns None if neither exists."""
    csv_path = data_dir / f"{stem}.csv"
    xlsx_path = data_dir / f"{stem}.xlsx"
    if csv_path.exists():
        return pd.read_csv(csv_path)
    if xlsx_path.exists():
        return pd.read_excel(xlsx_path, sheet_name=0, engine="openpyxl")
    return None


# --- Load core tables (CSV preferred, then .xlsx) ---
trans15 = load_table(DATA_DIR, "student_transcript_list15")
trans16 = load_table(DATA_DIR, "student_transcript_list16")
if trans15 is None and trans16 is None:
    raise FileNotFoundError("Need at least one of student_transcript_list15/16 (csv or xlsx)")
trans = pd.concat([df for df in [trans15, trans16] if df is not None], ignore_index=True)

perf15 = load_table(DATA_DIR, "fact_student_academic_performance_list15")
perf16 = load_table(DATA_DIR, "fact_student_academic_performance_list16")
perf = pd.concat([df for df in [perf15, perf16] if df is not None], ignore_index=True) if (perf15 is not None or perf16 is not None) else None

att15 = load_table(DATA_DIR, "student_attendance_list15")
att16 = load_table(DATA_DIR, "student_attendance_list16")
att = pd.concat([df for df in [att15, att16] if df is not None], ignore_index=True) if (att15 is not None or att16 is not None) else None

pay15 = load_table(DATA_DIR, "student_payments_list15")
pay16 = load_table(DATA_DIR, "student_payments_list16")
pay = pd.concat([df for df in [pay15, pay16] if df is not None], ignore_index=True) if (pay15 is not None or pay16 is not None) else None

spon15 = load_table(DATA_DIR, "student_sponsorships_list15")
spon16 = load_table(DATA_DIR, "student_sponsorships_list16")
spon = pd.concat([df for df in [spon15, spon16] if df is not None], ignore_index=True) if (spon15 is not None or spon16 is not None) else None

schools = load_table(DATA_DIR, "student_high_schools_all")

# Use empty DataFrames if a source is missing so downstream aggs/merges still run
if att is None:
    att = pd.DataFrame(columns=["REG_NO", "SEMESTER_INDEX", "DATE", "STATUS"])
if pay is None:
    pay = pd.DataFrame(columns=["REG_NO", "SEMESTER_INDEX", "PAYMENT_ID", "PAYMENT_DATE", "PAYMENT_STATUS", "AMOUNT_UGX", "PAYMENT_METHOD", "CHANNEL", "PAYMENT_SOURCE"])
if spon is None:
    spon = pd.DataFrame(columns=["REG_NO", "SEMESTER_INDEX", "SPONSOR_PAYMENT_ID", "STATUS", "AMOUNT_SPONSORED_UGX", "SPONSOR_NAME", "SCHOLARSHIP_TYPE", "DISBURSEMENT_DATE"])

(trans.shape, perf.shape if perf is not None else None, att.shape, pay.shape, spon.shape, None if schools is None else schools.shape)

((53457, 21), (53457, 15), (3204919, 5), (187402, 15), (10944, 11), (10000, 8))

In [ ]:
# --- Parse dates ---
att["DATE"] = pd.to_datetime(att["DATE"], errors="coerce")
pay["PAYMENT_DATE"] = pd.to_datetime(pay["PAYMENT_DATE"], errors="coerce")
spon["DISBURSEMENT_DATE"] = pd.to_datetime(spon["DISBURSEMENT_DATE"], errors="coerce")

# Keep only columns needed downstream (helps performance)
trans_cols = [
    "REG_NO","ACC_NO","PROGRAM","ACADEMIC_YEAR","SEMESTER","SEMESTER_INDEX",
    "CREDITS_ATTEMPTED","QUALITY_POINTS","COURSES_COUNT","PASSED_COURSES","FAILED_COURSES",
    "FCW_COUNT","FEX_COUNT","MEX_COUNT","CREDITS_PASSED","CREDITS_FAILED",
    "SEMESTER_GPA","CUM_CREDITS_ATTEMPTED","CUM_CREDITS_PASSED","CUM_QUALITY_POINTS","CGPA",
]
trans = trans[[c for c in trans_cols if c in trans.columns]].copy()

if perf is not None:
    perf_cols = ["REG_NO","SEMESTER_INDEX","COURSES_REGISTERED","TOTAL_CREDITS","QUALITY_POINTS","PASSED_COURSES","FAILED_COURSES","FCW_COUNT","FEX_COUNT","MEX_COUNT","SEMESTER_GPA"]
    perf = perf[[c for c in perf_cols if c in perf.columns]].copy()

# Normalization helpers
if "STATUS" in att.columns:
    att["STATUS"] = att["STATUS"].astype(str).str.upper().str.strip()
if "PAYMENT_STATUS" in pay.columns:
    pay["PAYMENT_STATUS"] = pay["PAYMENT_STATUS"].astype(str).str.upper().str.strip()
if "STATUS" in spon.columns:
    spon["STATUS"] = spon["STATUS"].astype(str).str.upper().str.strip()

att.head(3)

,REG_NO,ACC_NO,DATE,SEMESTER_INDEX,STATUS
0,S24B33/010,B29465,2024-09-10,1,PRESENT
1,S24B33/010,B29465,2024-09-11,1,PRESENT
2,S24B33/010,B29465,2024-09-12,1,PRESENT


### Optional: students list and academic progression (from CSV or Excel)

When available, we load `students_list15`/`students_list16` and `academic_progression_list15`/`academic_progression_list16` (CSV preferred, then .xlsx) to enrich the modeling table with student-level attributes and retake/attempt counts per semester.

In [ ]:
# Optional: students list (one row per student; adds STUDENT_LIST, TOTAL_REGISTRATIONS if present)
students_list15 = load_table(DATA_DIR, "students_list15")
students_list16 = load_table(DATA_DIR, "students_list16")
students_list = None
if students_list15 is not None or students_list16 is not None:
    parts = []
    if students_list15 is not None:
        s15 = students_list15.copy()
        s15["STUDENT_LIST"] = "LIST15"
        parts.append(s15)
    if students_list16 is not None:
        s16 = students_list16.copy()
        s16["STUDENT_LIST"] = "LIST16"
        parts.append(s16)
    students_list = pd.concat(parts, ignore_index=True)

# Optional: academic progression (aggregate retake/attempt counts per student-semester)
prog15 = load_table(DATA_DIR, "academic_progression_list15")
prog16 = load_table(DATA_DIR, "academic_progression_list16")
progression_sem = None
if prog15 is not None or prog16 is not None:
    prog = pd.concat([df for df in [prog15, prog16] if df is not None], ignore_index=True)
    if "REG_NO" in prog.columns and "SEMESTER_INDEX" in prog.columns:
        agg_kw = {"progression_count": ("REG_NO", "count")}
        if "RETAKE_FLAG" in prog.columns:
            agg_kw["retake_count_semester"] = ("RETAKE_FLAG", "sum")
        if "ATTEMPT_NUMBER" in prog.columns:
            agg_kw["max_attempt_semester"] = ("ATTEMPT_NUMBER", "max")
        progression_sem = prog.groupby(["REG_NO", "SEMESTER_INDEX"], as_index=False).agg(**agg_kw)
(None if students_list is None else students_list.shape, None if progression_sem is None else progression_sem.shape)

((10000, 37), (53075, 5))

## Attendance features (by student-semester)

We aggregate attendance events into rates per semester:
- `present_rate`, `late_rate`, `absent_rate`
- `attendance_days`, `attendance_span_days`

In [ ]:
def attendance_agg(att: pd.DataFrame) -> pd.DataFrame:
    a = att.dropna(subset=["REG_NO", "SEMESTER_INDEX"]).copy()
    a = a[a["DATE"].notna()].copy()

    # Map common status values
    status = a["STATUS"].replace({"P": "PRESENT", "A": "ABSENT", "L": "LATE"})
    a["_present"] = (status == "PRESENT").astype(int)
    a["_late"] = (status == "LATE").astype(int)
    a["_absent"] = (status == "ABSENT").astype(int)

    g = a.groupby(["REG_NO", "SEMESTER_INDEX"], as_index=False)
    out = g.agg(
        attendance_days=("DATE", "nunique"),
        present_days=("_present", "sum"),
        late_days=("_late", "sum"),
        absent_days=("_absent", "sum"),
        first_att_date=("DATE", "min"),
        last_att_date=("DATE", "max"),
    )

    out["present_rate"] = out["present_days"] / out["attendance_days"].replace({0: np.nan})
    out["late_rate"] = out["late_days"] / out["attendance_days"].replace({0: np.nan})
    out["absent_rate"] = out["absent_days"] / out["attendance_days"].replace({0: np.nan})
    out["attendance_span_days"] = (out["last_att_date"] - out["first_att_date"]).dt.days + 1

    return out.drop(columns=["first_att_date", "last_att_date"])


att_sem = attendance_agg(att)
att_sem.head()

,REG_NO,SEMESTER_INDEX,attendance_days,present_days,late_days,absent_days,present_rate,late_rate,absent_rate,attendance_span_days
0,AKM23B11/136,1,79,73,4,2,0.924051,0.050633,0.025316,111
1,AKM23B11/136,2,80,73,2,5,0.912500,0.025000,0.062500,110
2,AKM23B11/136,3,79,75,3,1,0.949367,0.037975,0.012658,109
3,AKM23B11/136,4,79,75,4,0,0.949367,0.050633,0.000000,111
4,AKM23B11/136,5,54,50,1,3,0.925926,0.018519,0.055556,74


## Payment features (by student-semester)

We aggregate payment transactions into:
- counts (`payment_txn_count`, success/fail counts)
- total amounts paid (success/fail totals)
- diversity of channels/methods
- `payment_span_days` (first→last payment date within the semester)

These approximate "tuition/fees payment trends" when we don’t have invoice/balance tables.

In [ ]:
def payments_agg(pay: pd.DataFrame) -> pd.DataFrame:
    p = pay.dropna(subset=["REG_NO", "SEMESTER_INDEX"]).copy()
    p = p[p["PAYMENT_DATE"].notna()].copy()

    status = p["PAYMENT_STATUS"].astype(str)
    p["_success"] = status.isin(["SUCCESS", "PAID", "COMPLETED"]).astype(int)
    p["_failed"] = status.isin(["FAILED", "DECLINED", "REVERSED", "CANCELLED"]).astype(int)

    p["AMOUNT_UGX"] = pd.to_numeric(p["AMOUNT_UGX"], errors="coerce")

    g = p.groupby(["REG_NO", "SEMESTER_INDEX"], as_index=False)
    out = g.agg(
        payment_txn_count=("PAYMENT_ID", "count"),
        payment_success_count=("_success", "sum"),
        payment_failed_count=("_failed", "sum"),
        total_paid_success=("AMOUNT_UGX", lambda s: float(s[p.loc[s.index, "_success"].astype(bool)].sum(skipna=True))),
        total_paid_failed=("AMOUNT_UGX", lambda s: float(s[p.loc[s.index, "_failed"].astype(bool)].sum(skipna=True))),
        distinct_methods=("PAYMENT_METHOD", "nunique"),
        distinct_channels=("CHANNEL", "nunique"),
        distinct_sources=("PAYMENT_SOURCE", "nunique"),
        first_pay_date=("PAYMENT_DATE", "min"),
        last_pay_date=("PAYMENT_DATE", "max"),
    )

    out["payment_success_rate"] = out["payment_success_count"] / out["payment_txn_count"].replace({0: np.nan})
    out["payment_span_days"] = (out["last_pay_date"] - out["first_pay_date"]).dt.days + 1

    return out.drop(columns=["first_pay_date", "last_pay_date"])


pay_sem = payments_agg(pay)
pay_sem.head()

,REG_NO,SEMESTER_INDEX,payment_txn_count,payment_success_count,payment_failed_count,total_paid_success,total_paid_failed,distinct_methods,distinct_channels,distinct_sources,payment_success_rate,payment_span_days
0,AKM23B11/136,1,1,1,0,3448601.0,0.0,1,1,1,1.000000,1
1,AKM23B11/136,2,2,1,1,1033347.0,2192720.0,1,2,1,0.500000,45
2,AKM23B11/136,3,5,2,3,939681.0,2304075.0,2,3,3,0.400000,31
3,AKM23B11/136,4,4,4,0,3404546.0,0.0,3,3,3,1.000000,47
4,AKM23B11/136,5,3,2,1,2207256.0,1002454.0,2,3,2,0.666667,51


## Sponsorship features (by student-semester)

We aggregate sponsor disbursements into:
- `has_sponsorship` (binary)
- total sponsored amount and approved amount
- distinct sponsor/type counts

This explicitly addresses your requirement to account for sponsored students.

In [ ]:
def sponsorship_agg(spon: pd.DataFrame) -> pd.DataFrame:
    s = spon.dropna(subset=["REG_NO", "SEMESTER_INDEX"]).copy()
    s["AMOUNT_SPONSORED_UGX"] = pd.to_numeric(s["AMOUNT_SPONSORED_UGX"], errors="coerce")

    s["_approved"] = s["STATUS"].isin(["APPROVED", "PAID", "SUCCESS"]).astype(int)

    g = s.groupby(["REG_NO", "SEMESTER_INDEX"], as_index=False)
    out = g.agg(
        sponsor_event_count=("SPONSOR_PAYMENT_ID", "count"),
        sponsor_approved_count=("_approved", "sum"),
        sponsor_amount_total=("AMOUNT_SPONSORED_UGX", "sum"),
        sponsor_amount_approved=("AMOUNT_SPONSORED_UGX", lambda x: float(x[s.loc[x.index, "_approved"].astype(bool)].sum(skipna=True))),
        sponsor_distinct_sponsors=("SPONSOR_NAME", "nunique"),
        sponsor_distinct_types=("SCHOLARSHIP_TYPE", "nunique"),
    )

    out["has_sponsorship"] = (out["sponsor_event_count"] > 0).astype(int)
    return out


spon_sem = sponsorship_agg(spon)
spon_sem.head()

,REG_NO,SEMESTER_INDEX,sponsor_event_count,sponsor_approved_count,sponsor_amount_total,sponsor_amount_approved,sponsor_distinct_sponsors,sponsor_distinct_types,has_sponsorship
0,AM25B32/004,1,1,1,954895,954895.0,1,1,1
1,AM25B32/004,2,1,0,1442106,0.0,1,1,1
2,AM25B32/004,3,1,1,2239452,2239452.0,1,1,1
3,AM25B32/007,1,1,1,607366,607366.0,1,1,1
4,AM25B32/007,2,1,1,2108574,2108574.0,1,1,1


## Past-performance features

To predict a student’s CGPA in semester *t*, we include lagged (previous-semester) performance:
- `prev_cgpa`, `prev_semester_gpa`
- prior failure flags and counts

This prevents target leakage by using information available **before** the semester being predicted.

In [ ]:
def add_lag_features(trans: pd.DataFrame) -> pd.DataFrame:
    t = trans.sort_values(["REG_NO", "SEMESTER_INDEX"]).copy()

    # Lag within each student
    for col in [
        "CGPA",
        "SEMESTER_GPA",
        "FAILED_COURSES",
        "PASSED_COURSES",
        "FCW_COUNT",
        "FEX_COUNT",
        "MEX_COUNT",
        "CREDITS_FAILED",
        "CREDITS_PASSED",
        "CUM_CREDITS_ATTEMPTED",
        "CUM_CREDITS_PASSED",
    ]:
        t[f"prev_{col.lower()}"] = t.groupby("REG_NO")[col].shift(1)

    t = t.rename(columns={
        "prev_cgpa": "prev_cgpa",
        "prev_semester_gpa": "prev_semester_gpa",
    })

    return t


trans_lag = add_lag_features(trans)
trans_lag[["REG_NO","SEMESTER_INDEX","CGPA","prev_cgpa","SEMESTER_GPA","prev_semester_gpa"]].head(10)

,REG_NO,SEMESTER_INDEX,CGPA,prev_cgpa,SEMESTER_GPA,prev_semester_gpa
18528,AKM23B11/136,1,4.10,NaN,4.10,NaN
18529,AKM23B11/136,2,3.97,4.10,3.83,4.10
18530,AKM23B11/136,3,4.25,3.97,4.76,3.83
18531,AKM23B11/136,4,4.29,4.25,4.41,4.76
18532,AKM23B11/136,5,4.34,4.29,4.55,4.41
18533,AKM23B11/136,6,4.31,4.34,4.17,4.55
0,AM25B32/002,1,3.28,NaN,3.28,NaN
1,AM25B32/002,2,2.84,3.28,2.48,3.28
2,AM25B32/002,3,3.00,2.84,3.34,2.48
3,AM25B32/003,1,4.92,NaN,4.92,NaN


## Build final modeling table

Join strategy:
- Base: `student_transcript_*` (contains target `CGPA`)
- Left-join aggregated attendance/payments/sponsorship by (`REG_NO`, `SEMESTER_INDEX`)
- Optional left-join `student_high_schools_all` by `REG_NO`
- Optional left-join `fact_student_academic_performance_*` by (`REG_NO`, `SEMESTER_INDEX`)

## Statistical validation: Do the chosen variables predict CGPA?

Before using the integrated table for modeling, we test whether the engineered variables are **statistically associated** with the target (CGPA). This provides evidence that the chosen features can aid prediction.

- **Numeric features:** Correlation with CGPA (Spearman, robust to non-normality) and a significance test (p-value). Variables with small p-values (e.g. &lt; 0.05) are considered statistically significant predictors.
- **Categorical features (e.g. PROGRAM):** Strength of association can be assessed via correlation of a numeric encoding with CGPA or via variance explained; the modeling notebook later uses permutation importance for a joint assessment.

Results are saved to `outputs/feature_cgpa_correlations.csv` for reporting.

In [ ]:
from scipy import stats

TARGET_COL = "CGPA"
# Exclude ids, target, and columns that are not predictors
exclude = {TARGET_COL, "REG_NO", "ACC_NO", "QUALITY_POINTS", "CUM_QUALITY_POINTS",
           "CUM_CREDITS_ATTEMPTED", "CUM_CREDITS_PASSED", "first_att_date", "last_att_date",
           "first_pay_date", "last_pay_date"}
candidate_cols = [c for c in model_df.columns if c not in exclude and model_df[c].notna().sum() > 100]

corr_list = []
for col in candidate_cols:
    y = model_df[TARGET_COL].astype(float)
    valid = y.notna()
    x = model_df.loc[valid, col]
    if pd.api.types.is_numeric_dtype(x):
        x_clean = x.astype(float)
        drop = x_clean.notna()
        if drop.sum() < 100:
            continue
        r, p = stats.spearmanr(x_clean.loc[drop], y.loc[drop], nan_policy="omit")
    else:
        # Categorical: encode as numeric then Spearman
        x_clean, _ = pd.factorize(x, use_na_sentinel=True)
        y_sub = y.loc[valid].values
        drop = (x_clean != -1) & np.isfinite(y_sub)
        if drop.sum() < 100:
            continue
        r, p = stats.spearmanr(x_clean[drop], y_sub[drop], nan_policy="omit")
    corr_list.append({"feature": col, "spearman_r": r, "p_value": p, "significant_005": p < 0.05})

corr_df = pd.DataFrame(corr_list).sort_values("p_value")
corr_df["abs_r"] = corr_df["spearman_r"].abs()
corr_df = corr_df.sort_values("abs_r", ascending=False).drop(columns=["abs_r"])

print("Association of each variable with CGPA (Spearman correlation, two-tailed p-value).")
print("significant_005 = True means the variable is statistically associated with CGPA at α=0.05.\n")
display(corr_df.head(40))

# Save for reporting
corr_df.to_csv(OUT_DIR / "feature_cgpa_correlations.csv", index=False)
OUT_DIR / "feature_cgpa_correlations.csv"

Association of each variable with CGPA (Spearman correlation, two-tailed p-value).
significant_005 = True means the variable is statistically associated with CGPA at α=0.05.



C:\Users\Admin\AppData\Local\Temp\ipykernel_15676\3297433467.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.spearmanr(x_clean.loc[drop], y.loc[drop], nan_policy="omit")


,feature,spearman_r,p_value,significant_005
14,prev_cgpa,0.890420,0.000000e+00,True
13,SEMESTER_GPA,0.872078,0.000000e+00,True
33,SEMESTER_GPA_perf,0.844658,0.000000e+00,True
27,QUALITY_POINTS_perf,0.787794,0.000000e+00,True
15,prev_semester_gpa,0.780976,0.000000e+00,True
9,FEX_COUNT,-0.644159,0.000000e+00,True
7,FAILED_COURSES,-0.638991,0.000000e+00,True
12,CREDITS_FAILED,-0.632829,0.000000e+00,True
31,FEX_COUNT_perf,-0.622399,0.000000e+00,True
29,FAILED_COURSES_perf,-0.617420,0.000000e+00,True


WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/outputs/feature_cgpa_correlations.csv')

In [ ]:
model_df = trans_lag.copy()
if perf is not None:
    model_df = model_df.merge(perf, on=["REG_NO", "SEMESTER_INDEX"], how="left", suffixes=("", "_perf"))
model_df = model_df.merge(att_sem, on=["REG_NO", "SEMESTER_INDEX"], how="left")
model_df = model_df.merge(pay_sem, on=["REG_NO", "SEMESTER_INDEX"], how="left")
model_df = model_df.merge(spon_sem, on=["REG_NO", "SEMESTER_INDEX"], how="left")

if schools is not None:
    model_df = model_df.merge(
        schools[["REG_NO", "HIGH_SCHOOL", "DISTRICT", "SCHOOL_TIER", "OWNERSHIP", "STUDENT_LIST"]],
        on="REG_NO",
        how="left",
        suffixes=("", "_school"),
    )

# Optional: students list (adds STUDENT_LIST if not from schools, plus TOTAL_REGISTRATIONS etc.)
if students_list is not None:
    sl = students_list.copy()
    # Normalize key column to REG_NO (Excel/CSV may use "REG NO", "Reg_No", etc.)
    reg_col = None
    for c in sl.columns:
        if str(c).strip().upper().replace(" ", "_") == "REG_NO":
            reg_col = c
            break
    if reg_col is not None:
        if reg_col != "REG_NO":
            sl = sl.rename(columns={reg_col: "REG_NO"})
        use_cols = [c for c in sl.columns if c == "REG_NO" or c not in model_df.columns]
        if len(use_cols) > 1:
            model_df = model_df.merge(sl[use_cols].drop_duplicates(subset=["REG_NO"], keep="first"), on="REG_NO", how="left")

# Optional: academic progression (retake/attempt counts per student-semester)
if progression_sem is not None:
    model_df = model_df.merge(progression_sem, on=["REG_NO", "SEMESTER_INDEX"], how="left")

# Fill sponsorship missingness (no row implies no sponsorship)
for col in [
    "sponsor_event_count","sponsor_approved_count","sponsor_amount_total","sponsor_amount_approved",
    "sponsor_distinct_sponsors","sponsor_distinct_types","has_sponsorship",
]:
    if col in model_df.columns:
        model_df[col] = model_df[col].fillna(0)

model_df.shape

(55779, 74)

In [ ]:
# Basic sanity checks
assert model_df[["REG_NO", "SEMESTER_INDEX"]].isna().any(axis=1).sum() == 0

# Target availability
model_df["CGPA"].isna().mean(), model_df["CGPA"].describe()

(np.float64(0.0),
 count    55779.000000
 mean         3.263903
 std          0.628692
 min          1.500000
 25%          2.810000
 50%          3.280000
 75%          3.720000
 max          5.000000
 Name: CGPA, dtype: float64)

In [ ]:
# Save modeling table
out_path = OUT_DIR / "modeling_table_student_semester.csv"
model_df.to_csv(out_path, index=False)
out_path

WindowsPath('d:/final-year-project-data-analysis/Other_Analysis/outputs/modeling_table_student_semester.csv')